# Day 27 — Final Project: Complete Data Engineering Pipeline

**Objective:** Build a production-grade end-to-end pipeline that combines every skill from the course:
- Medallion architecture (Bronze/Silver/Gold)
- Schema enforcement and data quality
- Window functions, joins, aggregations
- Delta Lake: MERGE, time travel, OPTIMIZE
- Structured Streaming with foreachBatch
- Analytical Gold layer with BI-ready output

**Scenario:** E-commerce platform — streaming order events, joined with product and customer dimensions, aggregated into daily sales reports.

In [ ]:
from pyspark.sql.functions import (
    col, to_timestamp, to_date, year, month, dayofmonth,
    trim, upper, when, coalesce, lit, current_timestamp,
    count, sum, avg, min, max, round, countDistinct,
    broadcast, row_number, dense_rank, lag, window
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, DateType
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import tempfile, os

LAKE = tempfile.mkdtemp(prefix="final_project_")
BRONZE = os.path.join(LAKE, "bronze", "orders")
SILVER = os.path.join(LAKE, "silver", "orders")
GOLD   = os.path.join(LAKE, "gold")
CKPT   = os.path.join(LAKE, "checkpoints")
SRC    = os.path.join(LAKE, "source")

for d in [BRONZE, SILVER, GOLD, CKPT, SRC]:
    os.makedirs(d, exist_ok=True)

print(f"Project lake: {LAKE}")

## Step 1: Dimension Tables (Static)

In [ ]:
# Product dimension
products = spark.createDataFrame([
    ("P001", "Laptop Pro",   "Electronics", 1299.99),
    ("P002", "Wireless Mouse","Electronics",   29.99),
    ("P003", "Standing Desk","Furniture",    499.99),
    ("P004", "Monitor 4K",   "Electronics",  699.99),
    ("P005", "Office Chair", "Furniture",    349.99),
], ["product_id", "product_name", "category", "list_price"])

# Customer dimension
customers = spark.createDataFrame([
    ("C001", "Alice Smith",   "EU", "Premium"),
    ("C002", "Bob Jones",     "US", "Standard"),
    ("C003", "Carol O'Brien", "EU", "Premium"),
    ("C004", "Dave Lee",      "APAC", "Standard"),
    ("C005", "Eve Chen",      "APAC", "Premium"),
], ["customer_id", "customer_name", "region", "tier"])

products.show()
customers.show()

## Step 2: Generate Raw Event Data (Source)

In [ ]:
import random
random.seed(42)

# Simulate 3 batches of raw order events (messy: nulls, duplicates, bad amounts)
def make_orders(n, date_prefix, batch):
    rows = []
    for i in range(n):
        cid = random.choice(["C001", "C002", "C003", "C004", "C005", None])
        pid = random.choice(["P001", "P002", "P003", "P004", "P005"])
        qty = random.randint(1, 5)
        price = str(round(random.uniform(10, 1500), 2)) if random.random() > 0.05 else "BAD"
        ts = f"{date_prefix} {random.randint(8,20):02d}:{random.randint(0,59):02d}:00"
        rows.append((f"ORD-{batch:02d}{i:03d}", cid, pid, qty, price, ts))
    return rows

all_rows = make_orders(20, "2024-06-01", 1) + \
           make_orders(20, "2024-06-02", 2) + \
           make_orders(20, "2024-06-03", 3)

# Add a few duplicates
all_rows += all_rows[:3]

raw_schema = StructType([
    StructField("order_id",    StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id",  StringType(), True),
    StructField("qty",         IntegerType(), True),
    StructField("unit_price",  StringType(), True),
    StructField("event_time",  StringType(), True),
])

raw_df = spark.createDataFrame(all_rows, schema=raw_schema)
raw_df.write.mode("overwrite").json(SRC)
print(f"Generated {raw_df.count()} raw events.")

## Step 3: Ingest to Bronze (Streaming)

In [ ]:
stream_raw = spark.readStream.schema(raw_schema).json(SRC)

def write_bronze(batch_df, batch_id):
    batch_df.withColumn("_ingested_at", current_timestamp()) \
            .withColumn("_batch_id", lit(batch_id)) \
            .write.format("delta") \
            .mode("append") \
            .save(BRONZE)
    print(f"[Bronze] batch={batch_id}, rows={batch_df.count()}")

q = stream_raw.writeStream \
    .foreachBatch(write_bronze) \
    .trigger(availableNow=True) \
    .option("checkpointLocation", os.path.join(CKPT, "bronze")) \
    .start()

q.awaitTermination(60)
print(f"Bronze rows: {spark.read.format('delta').load(BRONZE).count()}")

## Step 4: Bronze → Silver (Batch ETL)

In [ ]:
bronze_df = spark.read.format("delta").load(BRONZE)

# Quality: cast, clean, validate
silver_df = bronze_df \
    .withColumn("unit_price", col("unit_price").cast(DoubleType())) \
    .withColumn("event_time", to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("order_date", to_date(col("event_time"))) \
    .filter(
        col("order_id").isNotNull() &
        col("customer_id").isNotNull() &
        col("unit_price").isNotNull() &
        (col("unit_price") > 0) &
        col("product_id").isNotNull()
    ) \
    .withColumn("total_amount", round(col("qty") * col("unit_price"), 2))

# Dedup on order_id (keep first)
w_dedup = Window.partitionBy("order_id").orderBy("_ingested_at")
silver_clean = silver_df.withColumn("_rn", row_number().over(w_dedup)) \
                        .filter(col("_rn") == 1) \
                        .drop("_rn", "_ingested_at", "_batch_id")

# Enrich with dimensions
silver_enriched = silver_clean \
    .join(broadcast(products),  on="product_id",  how="left") \
    .join(broadcast(customers), on="customer_id", how="left")

print(f"Silver rows (after quality + dedup): {silver_enriched.count()}")
silver_enriched.select("order_id", "customer_name", "product_name",
                        "qty", "total_amount", "order_date", "region").show(5)

In [ ]:
# Write Silver as partitioned Delta table
silver_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_date") \
    .save(SILVER)

print("Silver written.")

## Step 5: Silver → Gold (Analytical Aggregations)

In [ ]:
silver = spark.read.format("delta").load(SILVER)

# Gold 1: Daily revenue by category and region
gold_daily = silver.groupBy("order_date", "category", "region") \
    .agg(
        count("order_id").alias("order_count"),
        round(sum("total_amount"), 2).alias("revenue"),
        round(avg("total_amount"), 2).alias("avg_order_value"),
        countDistinct("customer_id").alias("unique_customers"),
    ) \
    .orderBy("order_date", "category", "region")

gold_daily.show(truncate=False)
gold_daily.write.format("delta").mode("overwrite") \
    .save(os.path.join(GOLD, "daily_revenue"))
print("Gold daily_revenue written.")

In [ ]:
# Gold 2: Customer LTV + ranking
w_clv = Window.partitionBy("region").orderBy(col("lifetime_value").desc())

gold_clv = silver.groupBy("customer_id", "customer_name", "region", "tier") \
    .agg(
        count("order_id").alias("total_orders"),
        round(sum("total_amount"), 2).alias("lifetime_value"),
        countDistinct("category").alias("categories"),
    ) \
    .withColumn("region_rank", dense_rank().over(w_clv))

gold_clv.orderBy(col("lifetime_value").desc()).show(truncate=False)
gold_clv.write.format("delta").mode("overwrite") \
    .save(os.path.join(GOLD, "customer_ltv"))
print("Gold customer_ltv written.")

In [ ]:
# Gold 3: Product sales ranking with day-over-day change
product_daily = silver.groupBy("order_date", "product_id", "product_name", "category") \
    .agg(
        count("order_id").alias("units_sold"),
        round(sum("total_amount"), 2).alias("revenue"),
    )

w_lag = Window.partitionBy("product_id").orderBy("order_date")

product_trend = product_daily \
    .withColumn("prev_day_revenue", lag("revenue", 1).over(w_lag)) \
    .withColumn("revenue_delta",
        round(col("revenue") - coalesce(col("prev_day_revenue"), lit(0.0)), 2)) \
    .orderBy("order_date", col("revenue").desc())

product_trend.show(truncate=False)
product_trend.write.format("delta").mode("overwrite") \
    .save(os.path.join(GOLD, "product_trend"))
print("Gold product_trend written.")

## Step 6: Delta Maintenance

In [ ]:
# OPTIMIZE Silver
spark.sql(f"OPTIMIZE delta.`{SILVER}`")

# Show history
DeltaTable.forPath(spark, SILVER).history() \
    .select("version", "timestamp", "operation") \
    .show()

# Time travel — verify version 0 data
print("Silver version 0 row count:",
    spark.read.format("delta").option("versionAsOf", 0).load(SILVER).count())

## Step 7: Business Questions Answered from Gold

In [ ]:
gold_rev = spark.read.format("delta").load(os.path.join(GOLD, "daily_revenue"))
gold_ltv = spark.read.format("delta").load(os.path.join(GOLD, "customer_ltv"))

# Q1: Best performing category by total revenue
print("=== Top categories ===")
gold_rev.groupBy("category").agg(sum("revenue").alias("total")) \
    .orderBy(col("total").desc()).show()

# Q2: Premium vs Standard customer LTV comparison
print("=== LTV by tier ===")
gold_ltv.groupBy("tier").agg(
    round(avg("lifetime_value"), 2).alias("avg_ltv"),
    count("customer_id").alias("customers")
).show()

# Q3: Which region drives the most revenue?
print("=== Revenue by region ===")
gold_rev.groupBy("region").agg(sum("revenue").alias("total")) \
    .orderBy(col("total").desc()).show()

---

## Day 27 Final Project Complete!

Skills demonstrated:

| Skill | Where used |
|---|---|
| Structured Streaming + foreachBatch | Raw → Bronze ingestion |
| Schema enforcement + casting | Bronze → Silver cleaning |
| Window function (row_number dedup) | Silver deduplication |
| Broadcast join | Enrichment with product/customer dims |
| groupBy + agg | All Gold tables |
| Window lag for delta calculation | Product trend |
| Dense rank per partition | Customer LTV ranking |
| Delta Lake MERGE / OPTIMIZE / time travel | Step 6 |
| Medallion architecture | Full pipeline structure |

**Next:** Day 28 — Full Course Review